# ToolCallingAgent com smolagents: Agentes com Ferramentas

> **Artigo:** [ToolCallingAgent com smolagents: Agentes com Ferramentas](https://iablog.github.io)  
> **Autor:** Sarah P. Lima  
> **Data:** 23/05/2026

Neste notebook vamos construir um assistente de agendamento médico completo usando o `ToolCallingAgent` da biblioteca smolagents. O agente será capaz de:

- Listar especialidades e médicos disponíveis
- Verificar horários livres de um médico
- Agendar consultas para pacientes
- Mostrar um resumo dos agendamentos feitos

## 1. Instalação

In [1]:
!pip install -q "smolagents[litellm]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.0 MB/s eta 0:00:00


## 2. Configuração do Modelo

Aqui usamos o Gemini 2.5 Flash via LiteLLM. Você precisa de uma `GOOGLE_API_KEY` salva nos Secrets do Colab (`🔑` no menu lateral).

In [2]:
from smolagents import ToolCallingAgent, LiteLLMModel
from google.colab import userdata

model = LiteLLMModel(
    model_id="gemini/gemini-2.5-flash",
    api_key=userdata.get("GOOGLE_API_KEY"),
    timeout=60
)

12:28:35 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
12:28:35 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


## 3. Dados da Clínica

`MEDICOS` e `AGENDA` simulam um banco de dados. `CONSULTAS_AGENDADAS` acumula os agendamentos feitos durante a sessão.

In [3]:
MEDICOS = {
    "Dr. Carlos Mendes": {"especialidade": "Cardiologia", "crm": "PE-12345"},
    "Dra. Ana Lima":     {"especialidade": "Dermatologia", "crm": "PE-67890"},
    "Dr. Pedro Souza":   {"especialidade": "Ortopedia",   "crm": "PE-11223"},
    "Dra. Julia Costa":  {"especialidade": "Pediatria",   "crm": "PE-44556"},
}

AGENDA = {
    "Dr. Carlos Mendes": ["08:00", "09:00", "14:00", "15:00"],
    "Dra. Ana Lima":     ["10:00", "11:00", "16:00"],
    "Dr. Pedro Souza":   ["08:00", "13:00", "14:00"],
    "Dra. Julia Costa":  ["09:00", "10:00", "11:00", "15:00"],
}

CONSULTAS_AGENDADAS = []

## 4. Definindo as Ferramentas com `@tool`

O decorator `@tool` transforma uma função Python comum em uma ferramenta que o agente pode chamar. Dois elementos são essenciais:

- **A docstring**: o agente lê isso para decidir *quando* e *como* usar cada ferramenta
- **As type annotations**: o agente usa isso para passar os argumentos no tipo correto

In [4]:
from smolagents import tool

@tool
def listar_especialidades() -> str:
    """
    Lista todas as especialidades médicas disponíveis na clínica
    e os respectivos médicos de cada especialidade.
    Use esta ferramenta quando o paciente perguntar quais médicos
    ou especialidades estão disponíveis.
    """
    resultado = "Especialidades disponíveis:\n"
    for medico, dados in MEDICOS.items():
        resultado += f"- {dados['especialidade']}: {medico}\n"
    return resultado


@tool
def verificar_horarios(nome_medico: str) -> str:
    """
    Retorna os horários disponíveis de um médico específico.
    Use esta ferramenta antes de agendar, para verificar
    se há horários livres.

    Args:
        nome_medico: Nome completo do médico conforme listado
                     em listar_especialidades().
    """
    if nome_medico not in AGENDA:
        return f"Médico '{nome_medico}' não encontrado. Use listar_especialidades() para ver os nomes corretos."
    horarios = AGENDA[nome_medico]
    if not horarios:
        return f"{nome_medico} não possui horários disponíveis no momento."
    return f"Horários disponíveis de {nome_medico}: {', '.join(horarios)}"


@tool
def agendar_consulta(nome_paciente: str, nome_medico: str, horario: str) -> str:
    """
    Agenda uma consulta para o paciente com o médico no horário indicado.
    Só chame esta ferramenta após confirmar com verificar_horarios()
    que o horário está disponível.

    Args:
        nome_paciente: Nome completo do paciente.
        nome_medico: Nome completo do médico.
        horario: Horário no formato HH:MM (ex: '09:00').
    """
    if nome_medico not in AGENDA:
        return f"Médico '{nome_medico}' não encontrado."
    if horario not in AGENDA[nome_medico]:
        return f"Horário {horario} não está disponível para {nome_medico}."

    AGENDA[nome_medico].remove(horario)
    especialidade = MEDICOS[nome_medico]["especialidade"]
    CONSULTAS_AGENDADAS.append({
        "paciente": nome_paciente,
        "medico": nome_medico,
        "especialidade": especialidade,
        "horario": horario
    })
    return (f"Consulta agendada com sucesso!\n"
            f"Paciente: {nome_paciente}\n"
            f"Médico: {nome_medico} ({especialidade})\n"
            f"Horário: {horario}")


@tool
def ver_agendamentos() -> str:
    """
    Mostra todas as consultas agendadas na sessão atual.
    Use no final do atendimento ou quando o paciente pedir
    um resumo do que foi marcado.
    """
    if not CONSULTAS_AGENDADAS:
        return "Nenhuma consulta agendada ainda."
    resultado = "Consultas agendadas:\n"
    for c in CONSULTAS_AGENDADAS:
        resultado += (f"- {c['paciente']} com {c['medico']} "
                      f"({c['especialidade']}) às {c['horario']}\n")
    return resultado

## 5. Criando o Agente

In [5]:
agente_clinica = ToolCallingAgent(
    tools=[listar_especialidades, verificar_horarios,
           agendar_consulta, ver_agendamentos],
    model=model
)

### Inspecionando o system prompt

O smolagents monta automaticamente um system prompt a partir das ferramentas. Cada ferramenta vira um bloco com nome, descrição e parâmetros — é por isso que a docstring importa.

In [6]:
print(agente_clinica.system_prompt)

You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.
To do so, you have been given access to some tools.

The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
This Action/Observation can repeat N times, you should take several steps when needed.

You can use the result of the previous action as input for the next action.
The observation will always be a string: it can represent a file, like "image_1.jpg".
Then you can use it as input for the next action. You can do it for instance as follows:

Observation: "image_1.jpg"

Action:
{
  "name": "image_transformer",
  "arguments": {"image": "image_1.jpg"}
}

To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:
Action:
{
  "name": "final_answer"

## 6. Conversa em Múltiplos Turnos

O `ToolCallingAgent` suporta conversas com memória usando `reset=False`. A partir da segunda mensagem, o agente lembra o que foi dito e feito anteriormente.

### Turno 1 — O que está disponível?

In [7]:
from IPython.display import Markdown

resposta1 = agente_clinica.run(
    "Olá! Preciso marcar uma consulta. "
    "Quais especialidades vocês têm disponíveis?"
)
Markdown(resposta1)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Olá! Preciso marcar uma consulta. Quais especialidades vocês têm disponíveis?                                   │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'listar_especialidades' with arguments: {}                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Especialidades disponíveis:
- Cardiologia: Dr. Carlos Mendes
- Dermatologia: Dra. Ana Lima
- Ortopedia: Dr. Pedro Souza
- Pediatria: Dra. Julia Costa

[Step 1: Duration 1.19 seconds| Input tokens: 1,623 | Output tokens: 53]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Temos as seguintes especialidades disponíveis:\n-      │
│ Cardiologia: Dr. Carlos Mendes\n- Dermatologia: Dra. Ana Lima\n- Ortopedia: Dr. Pedro Souza\n- Pediatria: Dra.  │
│ Julia Costa\n\nQual especialidade ou médico você gostaria de agendar?'}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Temos as seguintes especialidades disponíveis:
- Cardiologia: Dr. Carlos Mendes
- Dermatologia: Dra. Ana Lima
- Ortopedia: Dr. Pedro Souza
- Pediatria: Dra. Julia Costa

Qual especialidade ou médico você gostaria de agendar?

Final answer: Temos as seguintes especialidades disponíveis:
- Cardiologia: Dr. Carlos Mendes
- Dermatologia: Dra. Ana Lima
- Ortopedia: Dr. Pedro Souza
- Pediatria: Dra. Julia Costa

Qual especialidade ou médico você gostaria de agendar?

[Step 2: Duration 1.32 seconds| Input tokens: 3,595 | Output tokens: 177]

Temos as seguintes especialidades disponíveis:
- Cardiologia: Dr. Carlos Mendes
- Dermatologia: Dra. Ana Lima
- Ortopedia: Dr. Pedro Souza
- Pediatria: Dra. Julia Costa

Qual especialidade ou médico você gostaria de agendar?

### Turno 2 — Verificando horários

In [8]:
resposta2 = agente_clinica.run(
    "Quero Cardiologia. Quais horários o Dr. Carlos Mendes tem disponível?",
    reset=False
)
Markdown(resposta2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Quero Cardiologia. Quais horários o Dr. Carlos Mendes tem disponível?                                           │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'verificar_horarios' with arguments: {'nome_medico': 'Dr. Carlos Mendes'}                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Horários disponíveis de Dr. Carlos Mendes: 08:00, 09:00, 14:00, 15:00

[Step 3: Duration 1.34 seconds| Input tokens: 6,049 | Output tokens: 294]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Dr. Carlos Mendes tem os seguintes horários            │
│ disponíveis: 08:00, 09:00, 14:00, 15:00. Qual horário você gostaria de agendar?'}                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Dr. Carlos Mendes tem os seguintes horários disponíveis: 08:00, 09:00, 14:00, 15:00. Qual horário 
você gostaria de agendar?

Final answer: Dr. Carlos Mendes tem os seguintes horários disponíveis: 08:00, 09:00, 14:00, 15:00. Qual horário 
você gostaria de agendar?

[Step 4: Duration 1.58 seconds| Input tokens: 9,047 | Output tokens: 442]

Dr. Carlos Mendes tem os seguintes horários disponíveis: 08:00, 09:00, 14:00, 15:00. Qual horário você gostaria de agendar?

### Turno 3 — Agendando

In [9]:
resposta3 = agente_clinica.run(
    "Pode marcar para Maria Silva às 14h.",
    reset=False
)
Markdown(resposta3)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Pode marcar para Maria Silva às 14h.                                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'agendar_consulta' with arguments: {'nome_paciente': 'Maria Silva', 'horario': '14:00',           │
│ 'nome_medico': 'Dr. Carlos Mendes'}                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Consulta agendada com sucesso!
Paciente: Maria Silva
Médico: Dr. Carlos Mendes (Cardiologia)
Horário: 14:00

[Step 5: Duration 1.36 seconds| Input tokens: 12,544 | Output tokens: 563]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Consulta agendada com sucesso para Maria Silva com Dr. │
│ Carlos Mendes (Cardiologia) às 14:00. Precisa de mais alguma coisa?'}                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Consulta agendada com sucesso para Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00. Precisa 
de mais alguma coisa?

Final answer: Consulta agendada com sucesso para Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00. Precisa 
de mais alguma coisa?

[Step 6: Duration 1.77 seconds| Input tokens: 16,486 | Output tokens: 653]

Consulta agendada com sucesso para Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00. Precisa de mais alguma coisa?

### Turno 4 — Agendando mais uma e pedindo resumo

Neste turno o agente recebe uma instrução que exige três ferramentas em sequência: verificar horários → agendar → mostrar resumo.

In [11]:
resposta4 = agente_clinica.run(
    "Sim! Quero também marcar Pediatria para o João Silva. "
    "Verifique os horários disponíveis e marque às 10h. "
    "Depois me mostre o resumo de tudo que foi agendado.",
    reset=False
)
Markdown(resposta4)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Sim! Quero também marcar Pediatria para o João Silva. Verifique os horários disponíveis e marque às 10h. Depois │
│ me mostre o resumo de tudo que foi agendado.                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'ver_agendamentos' with arguments: {}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Consultas agendadas:
- Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00
- João Silva com Dra. Julia Costa (Pediatria) às 10:00

[Step 10: Duration 1.63 seconds| Input tokens: 31,547 | Output tokens: 1,038]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Certo! Aqui está o resumo de tudo que foi agendado:\n- │
│ Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00\n- João Silva com Dra. Julia Costa (Pediatria) às      │
│ 10:00'}                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Certo! Aqui está o resumo de tudo que foi agendado:
- Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00
- João Silva com Dra. Julia Costa (Pediatria) às 10:00

Final answer: Certo! Aqui está o resumo de tudo que foi agendado:
- Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00
- João Silva com Dra. Julia Costa (Pediatria) às 10:00

[Step 11: Duration 1.71 seconds| Input tokens: 37,609 | Output tokens: 1,166]

Certo! Aqui está o resumo de tudo que foi agendado:
- Maria Silva com Dr. Carlos Mendes (Cardiologia) às 14:00
- João Silva com Dra. Julia Costa (Pediatria) às 10:00

## 7. Verificando o Estado Real

Confirmando que as ferramentas realmente modificaram o sistema — os horários foram removidos da `AGENDA` e os registros foram adicionados a `CONSULTAS_AGENDADAS`.

In [12]:
print("CONSULTAS AGENDADAS:", CONSULTAS_AGENDADAS)
print()
print("AGENDA Dr. Carlos Mendes:", AGENDA["Dr. Carlos Mendes"])  # 14:00 deve ter sido removido
print("AGENDA Dra. Julia Costa: ", AGENDA["Dra. Julia Costa"])   # 10:00 deve ter sido removido

CONSULTAS AGENDADAS: [{'paciente': 'Maria Silva', 'medico': 'Dr. Carlos Mendes', 'especialidade': 'Cardiologia', 'horario': '14:00'}, {'paciente': 'João Silva', 'medico': 'Dra. Julia Costa', 'especialidade': 'Pediatria', 'horario': '10:00'}]

AGENDA Dr. Carlos Mendes: ['08:00', '09:00', '15:00']
AGENDA Dra. Julia Costa:  ['09:00', '11:00', '15:00']


## 8. O Papel da Docstring

A docstring de cada ferramenta é instrução para o agente. Veja a diferença:

In [15]:
# Versão ruim — o agente não sabe quando nem como usar
# @ tool
# def verificar_horarios_ruim(nome_medico: str) -> str:
#    """Retorna horários."""
#    return ""

# Versão boa — o agente sabe quando usar e como passar o argumento
@tool
def verificar_horarios_boa(nome_medico: str) -> str:
    """
    Retorna os horários disponíveis de um médico específico.
    Use esta ferramenta antes de agendar, para verificar
    se há horários livres.

    Args:
        nome_medico: Nome completo do médico conforme listado
                     em listar_especialidades().
    """
    return ""

# A referência a listar_especialidades() no campo Args ensina o agente
# a encadear as ferramentas na ordem certa:
# 1. listar_especialidades()
# 2. verificar_horarios()
# 3. agendar_consulta()
print("Docstrings vagas causam chamadas erradas ou na ordem errada.")

Docstrings vagas causam chamadas erradas ou na ordem errada.


## Recursos Adicionais

- [Documentação oficial do smolagents — ToolCallingAgent](https://huggingface.co/docs/smolagents/reference/agents#smolagents.ToolCallingAgent)
- [Documentação do decorator @tool](https://huggingface.co/docs/smolagents/reference/tools#smolagents.tool)